### 복습
- 가전 폴더 안에 모든 데이터파일을 로드해서 하나의 데이터프레임으로 생성
- 감정에 대한 데이터들이 3개 분류 -> 2개의 분류로 변경(부정, 중립 -> 부정)
- 감정 데이터가 없는 데이터들은 따로 저장
- train, test의 비율은 8:2
- Dataset을 기존의 Dataset 구성과 같이 작업
- RawText 데이터를 이용하여 감정분석 모델을 생성
- SBERT 모델을 이용하여 임베딩
- 다중퍼셉트론의 모델을 이용하여 감정 분석 (Linear -> ReLU -> DropOut -> Linear)
- 검증 데이터를 이용하여 정확도와 f1_score 확인
- 감정 데이터가 없는 RawText에서 sample(10)를 출력하여 감정 예측

- 다중퍼셉트론 모델이 아닌 머신러닝 모델 (SVC)을 이용하여 감정 분석 예측

In [1]:
import os
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 문자 정규화 함수 정의
def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [3]:
# 파일의 목록을 로드 -> 목록을 기준으로 데이터를 로드 -> 단순 결합
file_path = '../data/가전/'
file_list = os.listdir(file_path)
file_list

['3-1.영상음향가전(76).json',
 '3-1.영상음향가전(77).json',
 '3-1.영상음향가전(78).json',
 '3-1.영상음향가전(79).json',
 '3-1.영상음향가전(80).json',
 '3-1.영상음향가전(81).json',
 '3-1.영상음향가전(82).json',
 '3-1.영상음향가전(83).json',
 '3-1.영상음향가전(84).json',
 '3-1.영상음향가전(85).json',
 '3-1.영상음향가전(86).json',
 '3-1.영상음향가전(87).json',
 '3-1.영상음향가전(88).json',
 '3-2.생활미용욕실가전(128).json',
 '3-2.생활미용욕실가전(129).json',
 '3-2.생활미용욕실가전(130).json',
 '3-2.생활미용욕실가전(131).json',
 '3-2.생활미용욕실가전(132).json',
 '3-2.생활미용욕실가전(133).json',
 '3-2.생활미용욕실가전(134).json',
 '3-2.생활미용욕실가전(135).json',
 '3-2.생활미용욕실가전(136).json',
 '3-2.생활미용욕실가전(137).json',
 '3-2.생활미용욕실가전(138).json',
 '3-2.생활미용욕실가전(139).json',
 '3-2.생활미용욕실가전(140).json',
 '3-3.주방가전(127).json',
 '3-3.주방가전(128).json',
 '3-3.주방가전(129).json',
 '3-3.주방가전(130).json',
 '3-3.주방가전(131).json',
 '3-3.주방가전(132).json',
 '3-3.주방가전(133).json',
 '3-3.주방가전(134).json',
 '3-3.주방가전(135).json',
 '3-3.주방가전(136).json',
 '3-3.주방가전(137).json',
 '3-3.주방가전(138).json',
 '3-3.주방가전(139).json',
 '3-4.계절가전(126).json',
 '3-4.계절가전(127)

In [4]:
# 로드한 데이터프레임을 누저긍로 결합하기 위해 빈 데이터프레임 생성
df = pd.DataFrame()

for file in file_list:
    # file : 파일명
    data = pd.read_json(file_path + file)
    # df에 단순 행 결합 -> df에 다시 대입
    df = pd.concat([df, data], axis=0)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [5]:
# 필요한 컬럼을 제외하고 나머지 컬럼을 무시
df = df[['RawText', 'GeneralPolarity']]

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RawText          4056 non-null   object 
 1   GeneralPolarity  3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 95.1+ KB


In [7]:
df.rename(columns={'GeneralPolarity' : 'label'}, inplace=True)

In [8]:
# RawText의 문자 정규화 사용
df['RawText'] = df['RawText'].map(normalize)

In [9]:
# RawText에서 길이가 1이하인 데이터를 제외
df = df.loc[df['RawText'].str.len() > 1]

In [10]:
# RawText에 중복 데이터가 존재할 수 있으니 중복 제거
df.drop_duplicates(subset='RawText', inplace=True)

In [11]:
# label이 결측치인 데이터는 따로 저장
na_df = df.loc[df['label'].isna(), ]
na_df

,RawText,label
13,귀에서 자꾸 빠져요.귀에 꼽는재 질이 미끄러운 재질이라 작은 소품이지만 재질만 바꾸...,NaN
43,아이를 출산한 기념으로 TV를 바꿨습니다. 그전에 쓰던 TV가 꽤나 무거워서 떨어지...,NaN
44,화면에 노이즈가 생깁니다. 저희 가족 중에 아무도 TV 화면을 건드리거나 하지 않았...,NaN
55,이번에 이사하면서 우리 따님께서 방에 TV가 있으면 좋겠다고 하여 방에서 사용할 T...,NaN
93,기존에 사용하던 무선이어폰이 오래되어서 배터리가 광탈하는 바람에 새로운 상품이 필요...,NaN
...,...,...
41,대용량이고 세척이 간편하다는 얘기에 구매를 했는데 저는 별로인 거 같아요...생각했...,NaN
44,요거 진짜 진짜 물건입니다. 처음엔 디자인이 너무 귀여워서 주문하게 되었는데요. 작...,NaN
68,지인의 추천으로 믿고 바로 구매를 해서 현재도 사용중입니다좀 더 많은 분들에게 도움...,NaN
76,다른 에어쿨러와 다르게 슬림한 디자인이 마음에 들어요.심플한 디자인 덕분에 집안 어...,NaN


In [12]:
# df에는 결측치를 제외
df = df.loc[~df['label'].isna()]
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 86.2+ KB


In [13]:
df.reset_index(drop=True, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3678 entries, 0 to 3677
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 57.6+ KB


In [14]:
# label의 데이터의 빈도수를 확인
df['label'].value_counts()

label
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [15]:
# label의 데이터들을 int형태로 변경
df['label'] = df['label'].astype(int)

In [16]:
df['label'].value_counts()

label
 1    2220
 0     944
-1     514
Name: count, dtype: int64

In [33]:
# 삼중 분류의 class를 이진 분류로 변경하기 위해 -1을 0으로 변경
df['label'] = df['label'].map(
    {
        -1 : 0,
        0 : 0,
        1 : 1
    }
)

In [ ]:
# /////삼중분류/////
df['label'] = df['label'].map(
    {
        -1 : 0,
        0 : 1,
        1 : 2
    }
)

In [27]:
df['label'].value_counts()

label
2    2220
1     944
0     514
Name: count, dtype: int64

In [28]:
# train, test로 분할
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)

In [29]:
train_df['label'].value_counts()

label
2    1776
1     755
0     411
Name: count, dtype: int64

In [30]:
model_name = 'BM-K/KoSimCSE-roberta-multitask'
sbert = SentenceTransformer(model_name)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [31]:
# 최대 토큰 길이 제한
sbert.max_seq_length = 128

In [32]:
class SBERTHead(Dataset):
    def __init__(self, texts, labels):
        with torch.inference_mode():
            self.emb = sbert.encode(texts, convert_to_tensor=True, normalize_embeddings=True)
        self.labels = torch.tensor(labels, dtype=torch.long)
        # self.texts = texts
        # self.labels2 = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.emb[idx], self.labels[idx]
        # getitem에서 임베딩 처리
        # res_emb = sbert.encode(self.texts[idx], convert_to_tensor=True, normalize_embeddings=True)
        # res_label = torch.tensor(self.labels2[idx], dtype=torch.long)
        # return res_emb, res_label

In [33]:
train_ds = SBERTHead(train_df['RawText'].tolist(), train_df['label'].tolist())
test_ds = SBERTHead(test_df['RawText'].tolist(), test_df['label'].tolist())

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256)

In [42]:
# 다중 퍼셉트론 구조의 분류 모델 생성
class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden = 256, num_classes = 2, dropout = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x):
        result = self.net(x)
        return result

In [35]:
# sbert모델에서 출력 차원의 개수를 확인
in_dim = sbert.get_sentence_embedding_dimension()
clf = MLPHead(in_dim)
# 손실 함수
crit = nn.CrossEntropyLoss()
# AdamW -> Adam 개선판
opt = optim.AdamW(clf.parameters(), lr=2e-4)

In [59]:
# ///3중분류///
in_dim = sbert.get_sentence_embedding_dimension()
clf = MLPHead(in_dim, num_classes=3, dropout=0.3)
# 손실 함수 -> 데이터의 불균형 문제를 해결하기 위해 class 별 가중치 변화를 각각 부여
# [0, 1, 2] -> 적은 데이터일수록 높은 변화
# 5.0 ~ 10.0 : 데이터의 비율이 굉장히 적은 경우
# 2.0 ~ 4.0 : 데이터의 비율이 적은 경우
# 1.0 : 가장 많은 데이터 비율
crit = nn.CrossEntropyLoss(weight=torch.tensor([5.0, 3.0, 1.0]))
# AdamW -> Adam 개선판
opt = optim.AdamW(clf.parameters(), lr=2e-4)

In [60]:
clf.train()

for epoch in range(10):
    total = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        logits = clf(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        total += loss.item() * x.size(0)
    print(f'epoch : {epoch}, loss : {round(total/len(train_df), 4)}')

epoch : 0, loss : 1.0835
epoch : 1, loss : 1.0454
epoch : 2, loss : 0.9938
epoch : 3, loss : 0.9345
epoch : 4, loss : 0.8768
epoch : 5, loss : 0.8234
epoch : 6, loss : 0.7827
epoch : 7, loss : 0.7517
epoch : 8, loss : 0.7285
epoch : 9, loss : 0.7099


In [61]:
# 정확도, f1_score 확인
clf.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for x, y in test_dl:
        logits = clf(x)
        pred = logits.argmax(dim=1).tolist()
        y_true += y.tolist()
        y_pred += pred

print('accuracy_score :', accuracy_score(y_true, y_pred))
print('f1_score :', f1_score(y_true, y_pred, average='macro'))

accuracy_score : 0.6521739130434783
f1_score : 0.6589864146695644


In [46]:
samples = na_df['RawText'].sample(10).tolist()
samples

['이발기는 처음 사서 제가 너무 기대가 컸나 봅니다.막상 사서 열어서 보니까 정말 잘 할 수 있을지 걱정이 앞서네요.일단 제품 구성은 뭐가 많이 들어 있어서 이런 게 다 필요한가 하는 생각까지 들게 풍성하네요.빗살 갭이 6종류나 제공돼서 정말 길이대로 다 들어 있고 헤어 스펀지랑 커트 빗이 추가 구성되어 있어 구성품이 아주 알차고 좋아요.티타늄 코팅이라 절삭력이 뛰어나다고 해서 머리카락에 살짝 사용해 보니 정말 잘 짜리기는 하는 거 같아요.제가 머리카락이 얇은 편이라 그런 거 같기도 하고 아직은 적응하는 중이에요.그런데 사용한 지 얼마 안 됐는데 금방 발열감을 느낄 정도로 날부분이 뜨거워져요.처음 사용하는데도 날부분이 이렇게 금방 뜨거워지는 걸 보니 그렇게 좋은 제품은 아닌가 싶어 실망이네요',
 '일반 다리미기만 쓰다가 스팀다리미가 간편하다고 해서 구매했어요.상품 받자마자 물 채우고 사용해 봤는데 간단하게 사용은 가능하지만 성능이 좋은 것 같지는 않아요.무게는 가볍다고 하지만 그래도 계속 한 손으로 들고 하려니까 손목에 무리가 가긴 하네요.처음 사용할 때 스팀이 안 나오고 버튼을 눌러도 소리만 나와 불량인 줄 알고 문의하니 물이 채워지는 데 시간이 걸려서 그렇다고 하네요.그래도 여러 번 버튼 눌러서 스팀은 나왔는데 스팀이 갑자기 세게 확 나오네요.간단하게 사용은 하겠지만 성능이 아주 좋거나 가볍게 사용하기 좋은 제품은 아닌 것 같아요.아무 걱정 안 하고 믿고 구매했는데 제 기대보다는 못해서 많이 실망했어요.',
 '사이즈가 너무 작아서 가족 많은 집은 힘들겠어요.이렇게 작은 줄은 사실 모르고 구매했네요. 1단 2단으로 되어 있는데 높이가 낮아서 큰 그릇은 아예 넣을 생각도 못 하겠네요.사이즈가 작아서 그런가 소음은 별로 없네요.그리고 완료되면 음성이 나오는 줄 알았는데 음성 안내가 전혀 없어요. 일단 우리 집은 두 식구라 상관없는데 4가족은 이거 한 번에 못 돌릴 것 같구요.그릇도 양식기 위주여야 할 것 같아요.이왕 살 거 큰 거로 살 걸 하는 생각이 들어

In [47]:
@torch.no_grad()
def predict_review(texts, batch_size=128):
    if isinstance(texts, str):
        texts = [texts]
    texts_norm = [normalize(t) for t in texts]

    sbert.eval()
    clf.eval()

    result = []

    for idx in range(0, len(texts), batch_size):
        batch_text = texts_norm[idx : idx + batch_size]
        embs = sbert.encode(batch_text, convert_to_tensor=True, normalize_embeddings=True)

        logits = clf(embs)
        probs = logits.softmax(dim=-1)
        preds = probs.argmax(dim=-1).tolist()
        id2label = {
            0 : '부정',
            1 : '중립',
            2 : '긍정'
        }
        for idx2, pred in enumerate(preds):
            prob = float(probs[idx2, pred])
            review = texts[idx + idx2]
            label = id2label[pred]
            result.append(
                {
                    'text' : review,
                    'prob' : prob,
                    'label' : label
                }
            )
    return result

In [48]:
predict_review(samples)

[{'text': '이발기는 처음 사서 제가 너무 기대가 컸나 봅니다.막상 사서 열어서 보니까 정말 잘 할 수 있을지 걱정이 앞서네요.일단 제품 구성은 뭐가 많이 들어 있어서 이런 게 다 필요한가 하는 생각까지 들게 풍성하네요.빗살 갭이 6종류나 제공돼서 정말 길이대로 다 들어 있고 헤어 스펀지랑 커트 빗이 추가 구성되어 있어 구성품이 아주 알차고 좋아요.티타늄 코팅이라 절삭력이 뛰어나다고 해서 머리카락에 살짝 사용해 보니 정말 잘 짜리기는 하는 거 같아요.제가 머리카락이 얇은 편이라 그런 거 같기도 하고 아직은 적응하는 중이에요.그런데 사용한 지 얼마 안 됐는데 금방 발열감을 느낄 정도로 날부분이 뜨거워져요.처음 사용하는데도 날부분이 이렇게 금방 뜨거워지는 걸 보니 그렇게 좋은 제품은 아닌가 싶어 실망이네요',
  'prob': 0.34331879019737244,
  'label': '중립'},
 {'text': '일반 다리미기만 쓰다가 스팀다리미가 간편하다고 해서 구매했어요.상품 받자마자 물 채우고 사용해 봤는데 간단하게 사용은 가능하지만 성능이 좋은 것 같지는 않아요.무게는 가볍다고 하지만 그래도 계속 한 손으로 들고 하려니까 손목에 무리가 가긴 하네요.처음 사용할 때 스팀이 안 나오고 버튼을 눌러도 소리만 나와 불량인 줄 알고 문의하니 물이 채워지는 데 시간이 걸려서 그렇다고 하네요.그래도 여러 번 버튼 눌러서 스팀은 나왔는데 스팀이 갑자기 세게 확 나오네요.간단하게 사용은 하겠지만 성능이 아주 좋거나 가볍게 사용하기 좋은 제품은 아닌 것 같아요.아무 걱정 안 하고 믿고 구매했는데 제 기대보다는 못해서 많이 실망했어요.',
  'prob': 0.3420632481575012,
  'label': '긍정'},
 {'text': '사이즈가 너무 작아서 가족 많은 집은 힘들겠어요.이렇게 작은 줄은 사실 모르고 구매했네요. 1단 2단으로 되어 있는데 높이가 낮아서 큰 그릇은 아예 넣을 생각도 못 하겠네요.사이즈가 작아서 그런가 소음은 별로 없네요.그리고 완료